<a href="https://colab.research.google.com/github/sawicka-byte/GWB43-groundwater-drought-DIPI/blob/main/01_preprocessing_SPI_SGI_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 Preprocessing of SPI and SGI

This notebook performs the first analytical step of the reproducible workflow.

Its purpose is to:
- load monthly groundwater-level and precipitation data,
- perform basic preprocessing,
- prepare time series for index calculation,
- export processed monthly SPI and SGI tables used in subsequent workflow steps.

## Inputs and outputs

### Input files
- `../data_input/groundwater_levels_monthly.csv`
- `../data_input/metadata_piezometers.csv`
- `../data_input/precipitation_monthly.csv`

### Output files
- `../data_processed/SPI_monthly.csv`
- `../data_processed/SGI_monthly.csv`

If local filenames differ, adjust the path definitions below.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

In [ ]:
repo_root = Path("..").resolve()

input_dir = repo_root / "data_input"
output_dir = repo_root / "data_processed"

groundwater_file = input_dir / "groundwater_levels_monthly.csv"
metadata_file = input_dir / "metadata_piezometers.csv"
precip_file = input_dir / "precipitation_monthly.csv"

print("Input directory:", input_dir)
print("Output directory:", output_dir)

In [ ]:
gw = pd.read_csv(groundwater_file)
meta = pd.read_csv(metadata_file)
prec = pd.read_csv(precip_file)

print("Groundwater data shape:", gw.shape)
print("Metadata shape:", meta.shape)
print("Precipitation data shape:", prec.shape)

In [ ]:
display(gw.head())
display(meta.head())
display(prec.head())

## Preprocessing notes

This step should standardize date fields, verify column names, inspect missing values, and prepare monthly time series in a consistent tabular format.

If required, file-specific column names can be harmonized in the cleaning section below.

In [ ]:
# Parse dates
gw["date"] = pd.to_datetime(gw["date"])
prec["date"] = pd.to_datetime(prec["date"])

# Sort values
gw = gw.sort_values(["piezometer_id", "date"]).reset_index(drop=True)
prec = prec.sort_values("date").reset_index(drop=True)

# Merge metadata into groundwater table
gw_clean = gw.merge(
    meta[["piezometer_id", "aquifer_group"]],
    on="piezometer_id",
    how="left"
)

print("Groundwater cleaned shape:", gw_clean.shape)
display(gw_clean.head())

print("Precipitation cleaned shape:", prec.shape)
display(prec.head())

In [ ]:
# ------------------------------------------------------------------
# SPI preprocessing
# Monthly precipitation table prepared for SPI calculation
# ------------------------------------------------------------------

spi_monthly = prec.copy()

# Optional: mean precipitation across all IMGW stations
station_cols = [col for col in spi_monthly.columns if col != "date"]
spi_monthly["precip_mean_all_stations"] = spi_monthly[station_cols].mean(axis=1)

print("SPI monthly table prepared.")
print("Stations:", station_cols)
display(spi_monthly.head())

In [ ]:
# ------------------------------------------------------------------
# SGI preprocessing
# Monthly groundwater table prepared for SGI calculation
# ------------------------------------------------------------------

sgi_monthly = gw_clean.copy()

# Optional cluster monthly mean
cluster_monthly_mean = (
    sgi_monthly
    .groupby(["date", "cluster_id"], as_index=False)["groundwater_level_m"]
    .mean()
    .rename(columns={"groundwater_level_m": "cluster_mean_gwl_m"})
)

sgi_monthly = sgi_monthly.merge(
    cluster_monthly_mean,
    on=["date", "cluster_id"],
    how="left"
)

print("SGI monthly table prepared.")
display(sgi_monthly.head())

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

spi_outfile = output_dir / "SPI_monthly.csv"
sgi_outfile = output_dir / "SGI_monthly.csv"

spi_monthly.to_csv(spi_outfile, index=False)
sgi_monthly.to_csv(sgi_outfile, index=False)

print("Saved:")
print("-", spi_outfile)
print("-", sgi_outfile)

## Reproducibility note

This notebook represents the preprocessing stage of the workflow. The exported SPI and SGI tables serve as intermediate analytical products for subsequent event-metric and DIPI-construction steps.